# Approach 5: TabTransformer — Transformers for Tabular Prediction

**TabTransformer** is a transformer architecture designed specifically for tabular **classification and regression** — not Q&A.

**Key innovation:** Each **categorical column** is treated as a token with a learned embedding. These embeddings are processed through a standard transformer encoder. Numerical columns are concatenated directly to the final representation.

```
Categorical cols → Embeddings → Transformer Encoder ─┐
                                                       ├→ MLP → Prediction
Numerical cols  → Concatenate directly ────────────────┘
```

**Why this matters:** The transformer encoder allows categorical values to **interact and contextualize each other** — e.g., the embedding for `department=Engineering` will be different depending on `promoted=yes`.

**Library:** `tab-transformer-pytorch` by lucidrains

In [ ]:
# !pip install tab-transformer-pytorch torch pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tab_transformer_pytorch import TabTransformer
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")

df = pd.read_csv('../data/employees.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Step 1: Prepare the data

TabTransformer separates:
- **Categorical columns** → integer-encoded → fed as token IDs
- **Numerical columns** → float tensors → concatenated after transformer

In [ ]:
# Task: predict whether an employee was promoted (binary classification)
TARGET = 'promoted'

CATEGORICAL_COLS = ['department']   # columns with discrete categories
NUMERICAL_COLS   = ['salary', 'years_experience', 'performance_score']  # continuous

# Encode categoricals as integers
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"{col} → {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Encode target
target_le = LabelEncoder()
y = target_le.fit_transform(df[TARGET])  # 'no'→0, 'yes'→1
print(f"\nTarget: {dict(zip(target_le.classes_, target_le.transform(target_le.classes_)))}")

# Scale numericals
scaler = StandardScaler()
X_num = scaler.fit_transform(df[NUMERICAL_COLS])
X_cat = df[[c + '_enc' for c in CATEGORICAL_COLS]].values

print(f"\nCategorical features shape: {X_cat.shape}")
print(f"Numerical features shape:   {X_num.shape}")

In [ ]:
# Train/test split
X_cat_train, X_cat_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_cat, X_num, y, test_size=0.25, random_state=42
)

# Convert to tensors
cat_train = torch.LongTensor(X_cat_train)
cat_test  = torch.LongTensor(X_cat_test)
num_train = torch.FloatTensor(X_num_train)
num_test  = torch.FloatTensor(X_num_test)
y_train_t = torch.FloatTensor(y_train)
y_test_t  = torch.FloatTensor(y_test)

print(f"Train: {len(y_train)} samples | Test: {len(y_test)} samples")

## Step 2: Define the TabTransformer

In [ ]:
# Number of unique categories per categorical column
categories = tuple(
    int(df[col + '_enc'].max() + 1) for col in CATEGORICAL_COLS
)
print(f"Category counts per column: {categories}")

model = TabTransformer(
    categories=categories,          # unique values per categorical column
    num_continuous=len(NUMERICAL_COLS),   # number of numerical columns
    dim=32,                         # embedding dimension per category token
    dim_out=1,                      # binary classification output
    depth=3,                        # number of transformer layers
    heads=4,                        # attention heads
    attn_dropout=0.1,
    ff_dropout=0.1,
    mlp_hidden_mults=(4, 2),        # MLP after transformer: hidden sizes as multiples
    mlp_act=nn.ReLU()
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTabTransformer parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)

## Step 3: Training loop

In [ ]:
EPOCHS = 100
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

train_dataset = TensorDataset(cat_train, num_train, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for cat_batch, num_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(cat_batch, num_batch).squeeze()
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 20 == 0:
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  Loss: {avg_loss:.4f}")

## Step 4: Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(cat_test, num_test).squeeze()
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).long().numpy()

print("Test Results:")
print(f"Accuracy: {accuracy_score(y_test, preds):.2%}")
print()
print(classification_report(y_test, preds, target_names=['not promoted', 'promoted']))

## Step 5: Inspect learned category embeddings

One of the most powerful features of TabTransformer: categorical embeddings are **contextualized** by attention — they encode relationships, not just identity.

In [ ]:
import torch.nn.functional as F

# Extract the raw category embeddings (before transformer)
# These live in the embedding layer
with torch.no_grad():
    dept_embeddings = model.embeds[0].weight  # shape: [num_depts, dim]

dept_names = label_encoders['department'].classes_
print(f"Department embeddings shape: {dept_embeddings.shape}")
print(f"Departments: {list(dept_names)}")
print()

# Compute pairwise cosine similarities
print("Cosine similarity between department embeddings (after training):")
print("(1.0 = identical, 0.0 = orthogonal, -1.0 = opposite)")
print()

sims = F.cosine_similarity(
    dept_embeddings.unsqueeze(0),
    dept_embeddings.unsqueeze(1),
    dim=2
)

sim_df = pd.DataFrame(sims.numpy(), index=dept_names, columns=dept_names)
print(sim_df.round(3).to_string())

In [ ]:
# Individual predictions with confidence
model.eval()
test_df = df.iloc[list(range(len(df)))[-5:]].reset_index(drop=True)

cat_sample = torch.LongTensor([[label_encoders['department'].transform([r['department']])[0]] for _, r in test_df.iterrows()])
num_sample = torch.FloatTensor(scaler.transform(test_df[NUMERICAL_COLS]))

with torch.no_grad():
    logits = model(cat_sample, num_sample).squeeze()
    probs  = torch.sigmoid(logits).numpy()

print("Sample predictions:")
for i, (_, row) in enumerate(test_df.iterrows()):
    prob = probs[i] if len(probs.shape) > 0 else float(probs)
    pred = 'promoted' if prob > 0.5 else 'not promoted'
    true = row['promoted']
    correct = '✓' if (pred == 'promoted') == (true == 'yes') else '✗'
    print(f"  {correct} {row['name']:20s} | dept: {row['department']:12s} | "
          f"salary: ${row['salary']:,} | true: {true:3s} | pred: {pred} ({prob:.2%})")

## Key Takeaways

| | TabTransformer |
|---|---|
| **Task** | Classification / Regression (not Q&A) |
| **Architecture** | Transformer encoder + MLP head |
| **Input** | Categorical tokens + numerical features |
| **Key advantage** | Categorical features attend to each other |
| **Best for** | Structured ML tasks: fraud detection, churn, pricing |

**Comparison to gradient boosting (XGBoost/LightGBM):** TabTransformer is competitive on large datasets (>10K rows) but gradient boosting is often better on small datasets. TabTransformer wins on datasets with many high-cardinality categoricals.

**Next:** See `06_nanogpt_from_scratch.ipynb` — build and train your own GPT from scratch!